In [2]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv("../outputs/cleaned_transactions.csv")

In [3]:
df_clean["client_id"].nunique()

1219

In [4]:
df_clean["merchant_id"].nunique()

7255

In [5]:
customer_summary = (
    df_clean.groupby("client_id")
    .agg(
        total_transactions=("id", "count"),
        total_spend=("amount", "sum"),
        avg_transaction=("amount", "mean"),
        refund_count=("refund_flag", "sum"),
        failed_transactions=("error_flag", "sum")
    )
    .reset_index()
)

In [6]:
customer_summary.head()

,client_id,total_transactions,total_spend,avg_transaction,refund_count,failed_transactions
0,0,57,2973.60,52.168421,3,1
1,1,34,1309.21,38.506176,0,1
2,2,41,1002.26,24.445366,5,0
3,3,28,1447.12,51.682857,0,2
4,4,37,2109.87,57.023514,0,0


In [7]:
customer_summary.shape

(1219, 6)

In [8]:
customer_summary["customer_type"] = np.where(
    customer_summary["total_spend"] >
    customer_summary["total_spend"].median(),
    "High Value",
    "Regular"
)

In [9]:
customer_summary["activity_level"] = pd.cut(
    customer_summary["total_transactions"],
    bins=[0,20,50,1000],
    labels=["Low","Medium","High"]
)

In [10]:
merchant_summary = (
    df_clean.groupby("merchant_id")
    .agg(
        total_transactions=("id","count"),
        total_revenue=("amount","sum"),
        avg_transaction=("amount","mean"),
        refund_count=("refund_flag","sum"),
        failed_transactions=("error_flag","sum")
    )
    .reset_index()
)

In [11]:
merchant_summary.shape
merchant_summary.head()

,merchant_id,total_transactions,total_revenue,avg_transaction,refund_count,failed_transactions
0,22,7,60.58,8.654286,0,0
1,23,1,44.13,44.130000,0,0
2,57,3,97.60,32.533333,0,1
3,91,1,85.51,85.510000,0,0
4,100,7,453.57,64.795714,0,0


In [12]:
monthly_kpi = (
    df_clean.groupby(
        ["year","month"]
    )
    .agg(
        total_transactions=("id","count"),
        total_spend=("amount","sum"),
        avg_transaction=("amount","mean"),
        refunds=("refund_flag","sum"),
        failed_transactions=("error_flag","sum")
    )
    .reset_index()
)

In [13]:
monthly_kpi.head()

,year,month,total_transactions,total_spend,avg_transaction,refunds,failed_transactions
0,2010,1,369,16288.24,44.141572,15,5
1,2010,2,352,15567.60,44.226136,18,6
2,2010,3,397,17843.79,44.946574,24,3
3,2010,4,387,13923.92,35.979121,24,7
4,2010,5,390,18839.66,48.306821,23,5


In [14]:
risk_summary = (
    df_clean.groupby("errors")
    .size()
    .reset_index(name="count")
)

In [15]:
risk_summary

,errors,count
0,Bad CVV,26
1,"Bad CVV,Insufficient Balance",1
2,Bad Card Number,31
3,Bad Expiration,18
4,Bad PIN,124
5,Bad Zipcode,6
6,Insufficient Balance,489
7,Technical Glitch,86


In [16]:
customer_summary.to_csv(
    "../outputs/customer_summary.csv",
    index=False
)

merchant_summary.to_csv(
    "../outputs/merchant_summary.csv",
    index=False
)

monthly_kpi.to_csv(
    "../outputs/monthly_kpi.csv",
    index=False
)

risk_summary.to_csv(
    "../outputs/risk_summary.csv",
    index=False
)

In [17]:
merchant_summary.shape

(7255, 6)

In [18]:
payment_summary = (
    df_clean.groupby("use_chip")
    .agg(
        total_transactions=("id","count"),
        total_spend=("amount","sum"),
        avg_transaction=("amount","mean"),
        refunds=("refund_flag","sum"),
        failures=("error_flag","sum")
    )
    .reset_index()
)

payment_summary

,use_chip,total_transactions,total_spend,avg_transaction,refunds,failures
0,Chip Transaction,18000,723267.31,40.181517,1030,261
1,Online Transaction,5776,331080.27,57.319991,18,132
2,Swipe Transaction,26224,1092727.67,41.668993,1455,388


In [19]:
payment_summary.to_csv(
    "../outputs/payment_summary.csv",
    index=False
)